<a href="https://colab.research.google.com/github/2003Himansh/BIG-DATA-ANALYSIS-USING-PYSPARK/blob/main/big_data_analysis_using_pyspark11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("BigData_California_Housing_Analysis") \
    .getOrCreate()

print("Spark Version:", spark.version)


Spark Version: 4.0.2


In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

# Load dataset
data = fetch_california_housing()

pdf = pd.DataFrame(data.data, columns=data.feature_names)
pdf["Price"] = data.target

# Convert Pandas → Spark
df = spark.createDataFrame(pdf)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [ ]:
df.show(5)

+------+--------+------------------+------------------+----------+------------------+--------+---------+-----+
|MedInc|HouseAge|          AveRooms|         AveBedrms|Population|          AveOccup|Latitude|Longitude|Price|
+------+--------+------------------+------------------+----------+------------------+--------+---------+-----+
|8.3252|    41.0| 6.984126984126984|1.0238095238095237|     322.0|2.5555555555555554|   37.88|  -122.23|4.526|
|8.3014|    21.0| 6.238137082601054|0.9718804920913884|    2401.0| 2.109841827768014|   37.86|  -122.22|3.585|
|7.2574|    52.0| 8.288135593220339| 1.073446327683616|     496.0|2.8022598870056497|   37.85|  -122.24|3.521|
|5.6431|    52.0|5.8173515981735155|1.0730593607305936|     558.0| 2.547945205479452|   37.85|  -122.25|3.413|
|3.8462|    52.0| 6.281853281853282|1.0810810810810811|     565.0|2.1814671814671813|   37.85|  -122.25|3.422|
+------+--------+------------------+------------------+----------+------------------+--------+---------+-----+
o

In [ ]:
df.printSchema()

root
 |-- MedInc: double (nullable = true)
 |-- HouseAge: double (nullable = true)
 |-- AveRooms: double (nullable = true)
 |-- AveBedrms: double (nullable = true)
 |-- Population: double (nullable = true)
 |-- AveOccup: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Price: double (nullable = true)



In [ ]:
total_rows = df.count()
print("Total Rows:", total_rows)

Total Rows: 20640


In [ ]:
print("Number of Partitions:", df.rdd.getNumPartitions())

Number of Partitions: 2


In [ ]:
from pyspark.sql.functions import avg

df.select(avg("Price")).show()

+------------------+
|        avg(Price)|
+------------------+
|2.0685581690891293|
+------------------+



In [ ]:
high_income = df.filter(df.MedInc > 5)
high_income.show(5)

+------+--------+------------------+------------------+----------+------------------+--------+---------+-----+
|MedInc|HouseAge|          AveRooms|         AveBedrms|Population|          AveOccup|Latitude|Longitude|Price|
+------+--------+------------------+------------------+----------+------------------+--------+---------+-----+
|8.3252|    41.0| 6.984126984126984|1.0238095238095237|     322.0|2.5555555555555554|   37.88|  -122.23|4.526|
|8.3014|    21.0| 6.238137082601054|0.9718804920913884|    2401.0| 2.109841827768014|   37.86|  -122.22|3.585|
|7.2574|    52.0| 8.288135593220339| 1.073446327683616|     496.0|2.8022598870056497|   37.85|  -122.24|3.521|
|5.6431|    52.0|5.8173515981735155|1.0730593607305936|     558.0| 2.547945205479452|   37.85|  -122.25|3.413|
|6.1183|    49.0| 5.869565217391305|1.2608695652173914|      86.0| 3.739130434782609|   37.82|  -122.29| 0.75|
+------+--------+------------------+------------------+----------+------------------+--------+---------+-----+
o

In [ ]:
df.groupBy("HouseAge").avg("Price").show()

+--------+------------------+
|HouseAge|        avg(Price)|
+--------+------------------+
|     8.0|1.9441458252427177|
|     7.0|1.9329603428571422|
|    49.0|2.2033588059701494|
|    29.0|1.9722780043383947|
|    47.0|1.9060105050505047|
|    42.0| 1.987726032608696|
|    44.0| 2.162334719101123|
|    35.0|2.0729895024271836|
|    18.0| 1.933547666666666|
|    39.0| 2.088905609756097|
|     1.0|             1.443|
|    37.0|  2.07360772811918|
|    34.0|2.1459396226415075|
|    25.0| 2.204141802120141|
|    36.0|2.0757394315545232|
|    41.0|1.9725377364864858|
|     4.0|2.2923513612565443|
|    23.0|2.0455474107142844|
|    50.0|2.1788024264705883|
|    45.0|2.1779802040816314|
+--------+------------------+
only showing top 20 rows


In [ ]:
from pyspark.sql.functions import col

df = df.withColumn("Price_in_Lakhs", col("Price") * 100)
df.show(5)

+------+--------+------------------+------------------+----------+------------------+--------+---------+-----+------------------+
|MedInc|HouseAge|          AveRooms|         AveBedrms|Population|          AveOccup|Latitude|Longitude|Price|    Price_in_Lakhs|
+------+--------+------------------+------------------+----------+------------------+--------+---------+-----+------------------+
|8.3252|    41.0| 6.984126984126984|1.0238095238095237|     322.0|2.5555555555555554|   37.88|  -122.23|4.526|452.59999999999997|
|8.3014|    21.0| 6.238137082601054|0.9718804920913884|    2401.0| 2.109841827768014|   37.86|  -122.22|3.585|             358.5|
|7.2574|    52.0| 8.288135593220339| 1.073446327683616|     496.0|2.8022598870056497|   37.85|  -122.24|3.521|352.09999999999997|
|5.6431|    52.0|5.8173515981735155|1.0730593607305936|     558.0| 2.547945205479452|   37.85|  -122.25|3.413|341.29999999999995|
|3.8462|    52.0| 6.281853281853282|1.0810810810810811|     565.0|2.1814671814671813|   37

In [ ]:
df.orderBy(df.Price.desc()).show(15)

+-------+--------+------------------+------------------+----------+------------------+--------+---------+-------+--------------+
| MedInc|HouseAge|          AveRooms|         AveBedrms|Population|          AveOccup|Latitude|Longitude|  Price|Price_in_Lakhs|
+-------+--------+------------------+------------------+----------+------------------+--------+---------+-------+--------------+
| 9.7194|     9.0| 8.306260575296108|0.9763113367174281|    1981.0|3.3519458544839256|   37.49|  -121.89|5.00001|       500.001|
| 4.0294|    23.0| 6.105747126436782|1.1839080459770115|     998.0|2.2942528735632184|   33.42|  -117.62|5.00001|       500.001|
| 8.3337|    24.0|             7.915|              1.06|    1081.0|            2.7025|   37.66|  -121.93|5.00001|       500.001|
| 3.8992|     5.0| 7.678861788617886|1.4593495934959348|     616.0|2.5040650406504064|   33.43|  -117.72|5.00001|       500.001|
|11.3421|     4.0| 8.115079365079366|0.9404761904761905|     830.0|3.2936507936507935|   37.82|  

In [ ]:
df = df.repartition(4)
print("Partitions After Repartition:", df.rdd.getNumPartitions())

Partitions After Repartition: 4


In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# Combine Features
assembler = VectorAssembler(
    inputCols=data.feature_names,
    outputCol="features"
)

data_ml = assembler.transform(df)
data_ml = data_ml.select("features", "Price")

# Train Test Split
train, test = data_ml.randomSplit([0.8, 0.2], seed=42)

# Model
lr = LinearRegression(featuresCol="features", labelCol="Price")
model = lr.fit(train)

# Predictions
predictions = model.transform(test)
predictions.select("features", "Price", "prediction").show(5)

+--------------------+-------+--------------------+
|            features|  Price|          prediction|
+--------------------+-------+--------------------+
|[0.4999,28.0,7.67...|5.00001|  0.8651646337805872|
|[0.536,36.0,12.25...|0.14999|  1.3175952653842913|
|[0.7054,43.0,2.22...|  1.375|  1.0168547308133853|
|[0.8054,52.0,2.90...|    0.6|  1.1297900394898974|
|[0.8585,15.0,4.82...|   0.45|-0.13951062486292187|
+--------------------+-------+--------------------+
only showing top 5 rows


In [ ]:
print("R2 Score:", model.summary.r2)
print("RMSE:", model.summary.rootMeanSquaredError)
print("MAE:", model.summary.meanAbsoluteError)

R2 Score: 0.6110795590042233
RMSE: 0.7167164317116623
MAE: 0.526683756918936
